# Assignment 06: Custom Loss Functions (100 points)

**Unit 06: Programming PyTorch | AI 310**

USAAIO Round 2 problems often require custom loss functions — compound losses that combine multiple objectives, physics-based losses for PINNs, or metric learning losses. This assignment trains you to implement and debug loss functions.

**Notation**:
- $\hat{y}$ = prediction, $y$ = target
- $\mathcal{L}$ = loss (scalar)
- All loss functions should return a scalar by default

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries.

---

## Part 1 (15 points, coding)

Implement **cross-entropy loss from scratch**. Do NOT use `nn.CrossEntropyLoss`, `F.cross_entropy`, `F.log_softmax`, or `F.nll_loss`.

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^B \log\left(\frac{e^{z_{i, y_i}}}{\sum_j e^{z_{i,j}}}\right)$$

Use the log-sum-exp trick for numerical stability:

$$\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}, \quad m = \max_j z_j$$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def my_cross_entropy(logits, labels):
    """
    Cross-entropy loss from scratch.
    
    Args:
        logits: (B, C) raw scores (NOT probabilities)
        labels: (B,) integer class labels
    
    Returns:
        scalar loss
    """
    pass

In [ ]:
""" END OF THIS PART """
torch.manual_seed(42)
logits = torch.randn(8, 5)
labels = torch.randint(0, 5, (8,))

my_loss = my_cross_entropy(logits, labels)
ref_loss = F.cross_entropy(logits, labels)
assert torch.allclose(my_loss, ref_loss, atol=1e-5), \
    f"Expected {ref_loss.item():.6f}, got {my_loss.item():.6f}"

# Test with extreme values (stability)
extreme_logits = torch.tensor([[1000.0, 0.0, 0.0], [0.0, 0.0, 1000.0]])
extreme_labels = torch.tensor([0, 2])
extreme_loss = my_cross_entropy(extreme_logits, extreme_labels)
assert not torch.isnan(extreme_loss) and not torch.isinf(extreme_loss), \
    "Loss should be stable for extreme logits"
print(f"Part 1 passed! Loss: {my_loss.item():.6f}")

---

## Part 2 (15 points, coding)

Implement **Huber loss** (smooth L1 loss) as an `nn.Module`.

$$\mathcal{L}_{\delta}(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{if } |y - \hat{y}| \leq \delta \\ \delta \cdot (|y - \hat{y}| - \frac{\delta}{2}) & \text{otherwise} \end{cases}$$

This loss is quadratic for small errors (like MSE) and linear for large errors (like L1), making it robust to outliers.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        pass
    
    def forward(self, y_pred, y_true):
        """
        Args:
            y_pred: (B,) or (B, D) predictions
            y_true: (B,) or (B, D) targets
        Returns:
            scalar mean loss
        """
        pass

In [ ]:
""" END OF THIS PART """
huber = HuberLoss(delta=1.0)
y_pred = torch.tensor([0.5, 2.0, 10.0])
y_true = torch.tensor([0.0, 0.0, 0.0])

loss = huber(y_pred, y_true)
# residuals: [0.5, 2.0, 10.0]
# |0.5| <= 1: 0.5 * 0.25 = 0.125
# |2.0| > 1: 1.0 * (2.0 - 0.5) = 1.5
# |10.0| > 1: 1.0 * (10.0 - 0.5) = 9.5
# mean = (0.125 + 1.5 + 9.5) / 3 = 3.7083...
expected = torch.tensor((0.125 + 1.5 + 9.5) / 3)
assert torch.allclose(loss, expected, atol=1e-4), \
    f"Expected {expected.item():.4f}, got {loss.item():.4f}"

# Gradient flow
x = torch.randn(10, requires_grad=True)
loss = huber(x, torch.zeros(10))
loss.backward()
assert x.grad is not None
print(f"Part 2 passed! Huber loss: {loss.item():.4f}")

---

## Part 3 (20 points, coding)

Implement **Focal Loss** for handling class imbalance.

$$\mathcal{L}_{\text{focal}} = -\frac{1}{B}\sum_i \alpha (1 - p_{i,y_i})^\gamma \log(p_{i,y_i})$$

where $p_{i,c} = \text{softmax}(z_i)_c$ is the predicted probability for class $c$, and $y_i$ is the true class.

When $\gamma = 0$, this reduces to standard cross-entropy (times $\alpha$). When $\gamma > 0$, easy examples (high $p_{y_i}$) get down-weighted.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0):
        pass
    
    def forward(self, logits, labels):
        """
        Args:
            logits: (B, C) raw scores
            labels: (B,) integer class labels
        Returns:
            scalar loss
        """
        pass

In [ ]:
""" END OF THIS PART """
focal = FocalLoss(gamma=2.0, alpha=1.0)
logits = torch.randn(16, 5)
labels = torch.randint(0, 5, (16,))

fl = focal(logits, labels)
assert fl.dim() == 0, "Loss must be scalar"
assert fl.item() > 0, "Loss must be positive"

# When gamma=0, focal loss = alpha * cross_entropy
focal_g0 = FocalLoss(gamma=0.0, alpha=1.0)
fl_g0 = focal_g0(logits, labels)
ce = F.cross_entropy(logits, labels)
assert torch.allclose(fl_g0, ce, atol=1e-4), \
    f"With gamma=0, focal should equal CE. Got {fl_g0.item():.4f} vs {ce.item():.4f}"

# Focal loss should be <= CE (it down-weights easy examples)
assert fl.item() <= ce.item() + 0.01, "Focal loss should be <= CE when gamma > 0"

# Gradient flow
logits_g = torch.randn(8, 3, requires_grad=True)
fl_g = focal(logits_g, torch.randint(0, 3, (8,)))
fl_g.backward()
assert logits_g.grad is not None
print(f"Part 3 passed! Focal: {fl.item():.4f}, CE: {ce.item():.4f}")

---

## Part 4 (25 points, coding)

Implement a **Compound PINN Loss** for the damped harmonic oscillator:

$$u''(t) + 2\zeta\omega_n u'(t) + \omega_n^2 u(t) = 0$$

with initial conditions $u(0) = 1$, $u'(0) = 0$.

Your loss function should combine:
- $\mathcal{L}_{\text{ODE}}$: mean squared ODE residual at collocation points
- $\mathcal{L}_{\text{IC}_1}$: $(u(0) - 1)^2$
- $\mathcal{L}_{\text{IC}_2}$: $(u'(0) - 0)^2$

Total: $\mathcal{L} = \mathcal{L}_{\text{ODE}} + \lambda_1 \mathcal{L}_{\text{IC}_1} + \lambda_2 \mathcal{L}_{\text{IC}_2}$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class DampedOscillatorLoss(nn.Module):
    def __init__(self, zeta=0.1, omega_n=2.0, lambda_ic=100.0):
        """
        Args:
            zeta: damping ratio
            omega_n: natural frequency
            lambda_ic: weight for initial condition losses
        """
        pass
    
    def forward(self, net, t_collocation):
        """
        Compute compound PINN loss.
        
        Args:
            net: nn.Module mapping (N, 1) -> (N, 1)
            t_collocation: (N, 1) time points, requires_grad=True
        
        Returns:
            total_loss: scalar
            loss_dict: dict with 'ode', 'ic1', 'ic2', 'total' keys
        """
        pass

In [ ]:
""" END OF THIS PART """
torch.manual_seed(42)
pinn = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
criterion = DampedOscillatorLoss(zeta=0.1, omega_n=2.0, lambda_ic=100.0)

t = torch.linspace(0, 5, 100).reshape(-1, 1)
t.requires_grad_(True)

total_loss, loss_dict = criterion(pinn, t)

assert total_loss.dim() == 0, "Total loss must be scalar"
assert 'ode' in loss_dict and 'ic1' in loss_dict and 'ic2' in loss_dict
assert total_loss.item() > 0

# Verify gradients flow to network
pinn.zero_grad()
total_loss.backward()
has_grads = all(p.grad is not None for p in pinn.parameters())
assert has_grads, "Gradients must flow to network parameters!"
print(f"Part 4 passed! ODE: {loss_dict['ode']:.4f}, IC1: {loss_dict['ic1']:.4f}, IC2: {loss_dict['ic2']:.4f}")

---

## Part 5 (25 points, coding)

**Train the PINN** from Part 4 to solve the damped harmonic oscillator.

Use $\zeta = 0.1$, $\omega_n = 2.0$, and train for 3000 iterations with Adam (lr=1e-3).

After training:
- The ODE residual should be small (< 0.1)
- $u(0)$ should be close to 1.0
- $u'(0)$ should be close to 0.0

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# Build PINN
# Set up loss, optimizer
# Training loop (3000 iterations)
# Print loss every 500 iterations

# YOUR CODE HERE

In [ ]:
""" END OF THIS PART """
# Test boundary conditions
with torch.no_grad():
    u_at_0 = pinn(torch.tensor([[0.0]])).item()

t_test = torch.tensor([[0.0]], requires_grad=True)
u_test = pinn(t_test)
u_prime_at_0 = torch.autograd.grad(u_test, t_test, torch.ones_like(u_test))[0].item()

print(f"u(0) = {u_at_0:.4f} (target: 1.0)")
print(f"u'(0) = {u_prime_at_0:.4f} (target: 0.0)")

assert abs(u_at_0 - 1.0) < 0.15, f"u(0) should be ~1.0, got {u_at_0}"
assert abs(u_prime_at_0) < 0.3, f"u'(0) should be ~0, got {u_prime_at_0}"
print("Part 5 passed!")